## Imports

In [1]:
%load_ext autoreload
%autoreload 2

import pickle
import pandas as pd
import numpy as np

import clip
import open_clip
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision.models import convnext_base, ConvNeXt_Base_Weights, resnet50, ResNet50_Weights, vit_b_32, ViT_B_32_Weights
import matplotlib.pyplot as plt

from deepfake_utils.datasets import DeepFakeDataset
from deepfake_utils.train import train_epoch, validate_epoch, train
from deepfake_utils.models import MyModel

from PIL import Image

Extension horovod.torch has not been built: /anaconda/envs/azureml_py38_PT_TF/lib/python3.10/site-packages/horovod/torch/mpi_lib_v2.cpython-310-x86_64-linux-gnu.so not found
If this is not expected, reinstall Horovod with HOROVOD_WITH_PYTORCH=1 to debug the build error.
Warning! MPI libs are missing, but python applications are still available.


In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [3]:
image_dir_path = 'Deepfake-Eval-2024/image-data'
model_type = 'ResNet-50-CLIP'

In [4]:
for model_type, model_type_ in [('ResNet', 'ResNet-50-pretrained'), ('ResNet-CLIP', 'ResNet-50-pretrained-clip'), ('ViT', 'ViT-b32-pretrained'), ('ViT-CLIP', 'ViT-b32-pretrained-clip'), ('ConvNeXt', 'ConvNeXt-base-pretrained'), ('ConvNeXt-CLIP', 'ConvNeXt-base-pretrained-clip')]:
    print(model_type_)

    train_data = DeepFakeDataset("image-metadata-train.csv", image_dir_path, model_type, is_train = True)
    train_data_loader = DataLoader(train_data, batch_size = 32, shuffle = True)
    val_data = DeepFakeDataset("image-metadata-val.csv", image_dir_path, model_type, is_train = False)
    val_data_loader = DataLoader(val_data, batch_size = 32, shuffle = False)
    
    model = MyModel(model_type_, device, 2)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
    train_loss_, train_auroc_, train_auprc_, train_acc_, val_loss_, val_auroc_, val_auprc_, val_acc_, best_epoch = train(1, train_data_loader, val_data_loader, model, nn.CrossEntropyLoss(), optimizer, device, True, False, 1)
    print("\n\n\n")

ResNet-50-pretrained
ResNet-50-pretrained
Epoch 1
- - - - - - - - - - - - - - - - - - - - - - - - - 
Training...
	Training Progress: 	[  320/ 1365]
	Training Progress: 	[  640/ 1365]
	Training Progress: 	[  960/ 1365]
	Training Progress: 	[ 1280/ 1365]
	Training Progress: 	[ 1376/ 1365]
Validating...
	Evaluation Progress: 	[  160/  292]
	Evaluation Progress: 	[  320/  292]
Training Error: 
	Loss: 0.019555	ROC AUC: 0.680833	PR AUC: 0.769112	Accuracy: 0.643223
Validation Error: 
	Loss: 0.563714	ROC AUC: 0.805909	PR AUC: 0.864166	Accuracy: 0.678082

Model weights from epoch 1




ResNet-50-pretrained-clip
ResNet-50-pretrained-clip
Epoch 1
- - - - - - - - - - - - - - - - - - - - - - - - - 
Training...
	Training Progress: 	[  320/ 1365]
	Training Progress: 	[  640/ 1365]
	Training Progress: 	[  960/ 1365]
	Training Progress: 	[ 1280/ 1365]
	Training Progress: 	[ 1376/ 1365]
Validating...
	Evaluation Progress: 	[  160/  292]
	Evaluation Progress: 	[  320/  292]
Training Error: 
	Loss: 0.0205